In [1]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-17-jdk-headless openjdk-17-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 47 not upgraded.
Need to get 120 MB of archives.
After this operation, 273 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openjdk-17-jre-headless amd64 17.0.21.0.3+1-1~24.04.1 [48.2 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 

In [2]:
!wget https://dlcdn.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz
!tar -xzf hadoop-3.5.0.tar.gz -C /content

--2026-11-13 12:08:11--  https://dlcdn.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz
Resolving dlcdn.apache.org (dlcdn.apache.org)... 151.101.2.132, 2a04:4e42::644
Connecting to dlcdn.apache.org (dlcdn.apache.org)|151.101.2.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 581280703 (554M) [application/x-gzip]
Saving to: ‘hadoop-3.5.0.tar.gz’

hadoop-3.5.0.tar.gz 100%[===================>] 554.35M  24.1MB/s    in 6.9s    

2026-11-13 12:08:30 (80.8 MB/s) - ‘hadoop-3.5.0.tar.gz’ saved [581280703/581280703]



In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["PATH"] += ":" + os.environ["HADOOP_HOME"] + "/bin"

In [4]:
!hadoop version

Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


In [5]:
from google.colab import files
uploaded = files.upload()

Saving map_reduce_test_docs.zip to map_reduce_test_docs.zip


In [6]:
import zipfile

zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as archive:
    documents = [
        name for name in archive.namelist()
        if name.startswith("map_reduce_test_docs/") and name.endswith(".txt")
    ]
    archive.extractall("/content", documents)

print(f"Extracted {len(documents)} text documents.")

Extracted 7 text documents.


In [7]:
!ls /content/map_reduce_test_docs

doc1.txt  doc2.txt  doc3.txt  doc4.txt	doc5.txt  doc6.txt  doc7.txt


In [8]:
%%writefile /content/KeywordMapper.java
import java.io.IOException;
import java.util.Locale;

import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.lib.input.FileSplit;

public class KeywordMapper
        extends Mapper<LongWritable, Text, Text, IntWritable> {

    private String keyword;
    private String fileName;
    private long rowNumber = 0;

    private final Text outputKey = new Text();
    private final IntWritable outputValue = new IntWritable();

    @Override
    protected void setup(Context context)
            throws IOException, InterruptedException {

        String searchTerm =
                context.getConfiguration().get("search.keyword");

        if (searchTerm == null || searchTerm.trim().isEmpty()) {
            throw new IOException("Please provide a search keyword.");
        }

        keyword = searchTerm.trim().toLowerCase(Locale.ROOT);

        FileSplit fileSplit = (FileSplit) context.getInputSplit();
        fileName = fileSplit.getPath().getName();
    }

    @Override
    protected void map(LongWritable offset, Text line, Context context)
            throws IOException, InterruptedException {

        rowNumber++;

        String lineText = line.toString().toLowerCase(Locale.ROOT);
        int position = lineText.indexOf(keyword);

        while (position >= 0) {
            outputKey.set(fileName + "|" + rowNumber);
            outputValue.set(position + 1);
            context.write(outputKey, outputValue);

            position = lineText.indexOf(keyword, position + 1);
        }
    }
}

Writing /content/KeywordMapper.java


In [9]:
%%writefile /content/KeywordReducer.java
import java.io.IOException;
import java.util.ArrayList;
import java.util.Collections;
import java.util.List;

import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Reducer;

public class KeywordReducer
        extends Reducer<Text, IntWritable, Text, Text> {

    @Override
    protected void reduce(Text documentAndRow,
                          Iterable<IntWritable> columns,
                          Context context)
            throws IOException, InterruptedException {

        List<Integer> sortedColumns = new ArrayList<>();

        for (IntWritable column : columns) {
            sortedColumns.add(column.get());
        }

        Collections.sort(sortedColumns);

        StringBuilder result = new StringBuilder();

        for (int i = 0; i < sortedColumns.size(); i++) {
            if (i > 0) {
                result.append(", ");
            }
            result.append(sortedColumns.get(i));
        }

        context.write(documentAndRow, new Text(result.toString()));
    }
}

Writing /content/KeywordReducer.java


In [10]:
%%writefile /content/KeywordSearchDriver.java
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class KeywordSearchDriver {

    public static void main(String[] args) throws Exception {

        if (args.length != 3) {
            System.err.println(
                "Usage: KeywordSearchDriver <input-folder> <output-folder> <keyword>"
            );
            System.exit(2);
        }

        if (args[2].trim().isEmpty()) {
            throw new IllegalArgumentException(
                "The search keyword cannot be empty."
            );
        }

        Configuration configuration = new Configuration();
        configuration.set("search.keyword", args[2].trim());

        // Run Hadoop locally in the Colab environment
        configuration.set("mapreduce.framework.name", "local");
        configuration.set("fs.defaultFS", "file:///");

        Job job = Job.getInstance(configuration, "Keyword location search");

        job.setJarByClass(KeywordSearchDriver.class);
        job.setMapperClass(KeywordMapper.class);
        job.setReducerClass(KeywordReducer.class);
        job.setNumReduceTasks(1);

        job.setMapOutputKeyClass(Text.class);
        job.setMapOutputValueClass(IntWritable.class);
        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(Text.class);

        // Keep each supplied text document in one split for row counting
        FileInputFormat.setMaxInputSplitSize(job, 1024L * 1024L);

        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}

Writing /content/KeywordSearchDriver.java


In [11]:
!mkdir -p /content/classes
!javac -cp "$(hadoop classpath)" -d /content/classes \
  /content/KeywordMapper.java \
  /content/KeywordReducer.java \
  /content/KeywordSearchDriver.java

[0.022s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.023s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [12]:
!jar -cf /content/keyword-search.jar -C /content/classes .

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [13]:
!hadoop jar /content/keyword-search.jar KeywordSearchDriver \
  /content/map_reduce_test_docs \
  /content/keyword_search_output \
  Africa

Exception in thread "main" java.lang.UnsupportedClassVersionError: KeywordSearchDriver has been compiled by a more recent version of the Java Runtime (class file version 65.0), this version of the Java Runtime only recognizes class file versions up to 61.0
	at java.base/java.lang.ClassLoader.defineClass1(Native Method)
	at java.base/java.lang.ClassLoader.defineClass(ClassLoader.java:1017)
	at java.base/java.security.SecureClassLoader.defineClass(SecureClassLoader.java:150)
	at java.base/java.net.URLClassLoader.defineClass(URLClassLoader.java:524)
	at java.base/java.net.URLClassLoader$1.run(URLClassLoader.java:427)
	at java.base/java.net.URLClassLoader$1.run(URLClassLoader.java:421)
	at java.base/java.security.AccessController.doPrivileged(AccessController.java:712)
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:420)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at java.base/j

In [14]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [15]:
!java -version
!javac -version

!rm -rf /content/classes /content/keyword-search.jar /content/keyword_search_output
!mkdir -p /content/classes

!/usr/lib/jvm/java-17-openjdk-amd64/bin/javac \
  -cp "$(hadoop classpath)" \
  -d /content/classes \
  /content/KeywordMapper.java \
  /content/KeywordReducer.java \
  /content/KeywordSearchDriver.java

!jar -cf /content/keyword-search.jar -C /content/classes .

openjdk version "17.0.21.0.3" 2026-10-24
OpenJDK Runtime Environment (build 17.0.21.0.3+1-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.0.3+1-1-24.04.1-Ubuntu, mixed mode, sharing)
javac 17.0.21.0.3


In [16]:
!hadoop jar /content/keyword-search.jar KeywordSearchDriver \
  /content/map_reduce_test_docs \
  /content/keyword_search_output \
  Africa

2026-11-13 13:10:38,897 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-11-13 13:10:39,062 INFO input.FileInputFormat: Total input files to process : 7
2026-11-13 13:10:39,123 INFO mapreduce.JobSubmitter: number of splits:7
2026-11-13 13:10:39,601 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1251284724_0001
2026-11-13 13:10:39,604 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-13 13:10:39,993 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-11-13 13:10:39,995 INFO mapreduce.Job: Running job: job_local1251284724_0001
2026-11-13 13:10:39,996 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-11-13 13:10:40,014 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-11-13 13:10:40,017 INFO output.FileOutputCommitter

In [17]:
!cat /content/keyword_search_output/part-r-00000


doc1.txt|1	64
doc1.txt|3	11, 95
doc2.txt|1	57
doc2.txt|3	32
doc2.txt|33	24
doc2.txt|58	5, 81
doc3.txt|1	38
doc3.txt|3	7
doc3.txt|40	146
doc3.txt|46	7
doc3.txt|48	112
doc3.txt|5	7
doc3.txt|50	142
doc3.txt|52	86
doc3.txt|54	44
doc5.txt|1	62
doc5.txt|23	121
doc5.txt|3	88
doc5.txt|38	27
doc5.txt|48	217
